# Séance 1 — Généralités, codes de permutation et de substitution

**Cours de référence** : Pascal Boyer, *Cryptographie 2013* (Université Paris 13),
chapitre 1 (« Généralités ») et début du chapitre 2 (« Exemples simples »).

## Objectifs de la séance

1. écrire la **définition axiomatique** d'un code et la vérifier sur des exemples ;
2. mettre en œuvre un **code de permutation** (la scytale) ;
3. mettre en œuvre un **code de substitution** (le code de César) ;
4. comprendre la **cryptanalyse par analyse de fréquences** : retrouver une clef
   sans aucune information secrète, en utilisant les régularités du français.

## Ce que vous rendez

Votre fichier `tp1_permutation.py` complété (neuf fonctions), et les réponses aux
« questions de rapport » en fin de notebook. Pour vérifier votre travail à tout
moment :

    python3 test_tp1_permutation.py

ou, directement depuis ce notebook, la cellule `verifier()` plus bas.


## 1. Rappels du cours

### Définition d'un code

Un code est la donnée de deux ensembles finis de mots (les mots **en clair** $\mathcal{P}$
et les mots **codés** $\mathcal{C}$), d'un ensemble fini $\mathcal{K}$ de **clefs**, et
pour chaque clef $k \in \mathcal{K}$ de deux règles :

* une règle de chiffrement $e_k \in \mathcal{E}$ qui transforme les mots en clair en mots codés ;
* une règle de déchiffrement $d_k \in \mathcal{D}$ qui fait l'opération inverse.

La propriété fondamentale, c'est que **le déchiffrement défait exactement le chiffrement** :

$$\forall x \in \mathcal{P},\qquad d_k\big(e_k(x)\big) = x.$$

Toutes les fonctions que vous allez écrire dans ce TP doivent respecter cette égalité.
Les épreuves automatiques la vérifient.

### Deux familles de codes

| Famille | Idée | Exemple du cours |
|---|---|---|
| Code de **permutation** | on ne change aucune lettre, on les déplace | scytale des Spartiates |
| Code de **substitution** | on remplace chaque lettre par une autre | code de César |

### Principe de Kerckhoffs

La sécurité d'un cryptosystème ne doit pas reposer sur le secret de son
fonctionnement, mais **seulement sur le secret de la clef**. C'est pourquoi on peut
étudier publiquement les codes de ce TP : tout le monde connaît le principe de la
scytale ou de César, et pourtant, quand la clef est secrète, faut-il encore savoir
la retrouver ! C'est précisément l'objet de la **cryptanalyse**.


## 2. Installer votre espace de travail

Exécutez la cellule suivante (`Maj + Entrée`) : elle retrouve le dépôt, charge les
outils du professeur et définit trois fonctions utiles :

* `verifier()` : lance les épreuves automatiques du TP et affiche le bilan ;
* `charger()` : recharge votre fichier `tp1_permutation.py` après modification ;
* `trouver(...)` : retrouve un fichier du dépôt.


In [ ]:
import os, sys, subprocess

def trouver_racine(depart):
    """Remonte les dossiers jusqu'à trouver la racine du dépôt."""
    while True:
        if os.path.exists(os.path.join(depart, "outils", "reference.py")):
            return depart
        parent = os.path.dirname(depart)
        if parent == depart:
            raise RuntimeError("racine du dépôt introuvable")
        depart = parent

RACINE = trouver_racine(os.getcwd())
if RACINE not in sys.path:
    sys.path.insert(0, RACINE)

def trouver(nom):
    """Retrouve un fichier du dépôt (son dossier de séance, peu importe où l'on est)."""
    for base, _, fichiers in os.walk(RACINE):
        if nom in fichiers:
            return os.path.join(base, nom)
    raise FileNotFoundError(nom)

DOSSIER = os.path.dirname(trouver("test_tp1_permutation.py"))

def verifier():
    """Lance les épreuves automatiques de la séance et affiche le bilan."""
    resultat = subprocess.run([sys.executable, "test_tp1_permutation.py"], cwd=DOSSIER,
                              capture_output=True, text=True)
    print(resultat.stdout.strip()[-2500:])

def executer(consigne, fonction):
    """Exécute une cellule d'exercice, avec un rappel si votre fonction manque encore."""
    try:
        return fonction()
    except NotImplementedError as erreur:
        print("Cette cellule attend une fonction de votre fichier :", consigne)
        print("   ->", erreur)
        print("   Écrivez la fonction, enregistrez le fichier .py, puis relancez la cellule.")
        return None

def charger():
    """Recharge votre fichier de TP (après l'avoir modifié, relancez cette cellule)."""
    import importlib
    if DOSSIER not in sys.path:
        sys.path.insert(0, DOSSIER)
    if "tp1_permutation" in sys.modules:
        return importlib.reload(sys.modules["tp1_permutation"])
    import importlib.util
    spec = importlib.util.spec_from_file_location("tp1_permutation", os.path.join(DOSSIER, "tp1_permutation.py"))
    module = importlib.util.module_from_spec(spec)
    sys.modules["tp1_permutation"] = module
    spec.loader.exec_module(module)
    return module

print("Racine du dépôt :", RACINE)
print("Dossier de la séance :", DOSSIER)


## 3. Première observation : à quoi ressemble le français ?

Les lettres françaises ne sont pas utilisées avec la même fréquence. La cellule
suivante affiche l'histogramme des lettres d'un long texte français : regardez la
place du **E**, du **S**, du **A**.


In [ ]:
from outils.texte_francais import barres, frequences, texte_reference, distance_au_francais, vraisemblance

reference = texte_reference()
print("Longueur du corpus de référence :", len(reference), "lettres")
print()
print(barres(frequences(reference), largeur=35))
print()
print("Indice de coïncidence du corpus :", round(__import__("outils.texte_francais", fromlist=["x"]).indice_de_coincidence(reference), 4))

**À retenir** : la lettre E représente à elle seule plus de 15 % du texte, tandis que
K, W et Z sont presque absents. C'est cette inégalité que la cryptanalyse exploite.

Regardez maintenant le message César intercepté (`MESSAGE_CESAR` dans votre fichier) :
son histogramme est celui du français... mais **décalé**. C'est déjà tout César.


In [ ]:
tp = charger()
print("Longueur du message intercepté :", len(tp.MESSAGE_CESAR), "caractères")
print()
print(barres(frequences(tp.MESSAGE_CESAR), largeur=35))

## 4. Exercice 1 — préparer et compter (fonctions A.1 et A.2)

Ouvrez `tp1_permutation.py` et écrivez `nettoyer_simple` puis `compter_lettres`.
La cellule suivante recharge votre fichier et le teste.


In [ ]:
tp = charger()

def _exercice():
    print("nettoyer_simple :", tp.nettoyer_simple("Bonjour, le monde !"))
    print("compter_lettres :", tp.compter_lettres("BONJOUR"))

executer("A.1 nettoyer_simple et A.2 compter_lettres", _exercice)
verifier()

## 5. La scytale (code de permutation)

Le texte est écrit en lignes de $t$ lettres, puis lu en colonnes. Exemple avec $t = 5$ :

```
J E V O U          lecture en colonnes :
S A T T E          JSNH + EADE + VTST + OTAE + UECR
N D S A C
H E T E R
```

La fonction `scytale_chiffrer` est déjà écrite pour vous dans `common.py` / `outils/reference.py` :
la partie intéressante est son **attaque**, à la question B.3.

### Expérience : les fréquences sont-elles modifiées ?

Une permutation ne change **aucune** lettre : elle ne fait que les déplacer. Vérifions-le.


In [ ]:
from outils.reference import scytale_chiffrer, scytale_dechiffrer, cesar_chiffrer, cesar_dechiffrer
from outils.reference import comptage

clair = "LEGRANDMESSAGERARRIVECETTEAPRESMIDI"
chiffre = scytale_chiffrer(clair, 7)
print("Clair   :", clair)
print("Chiffré :", chiffre)
print("Déchiffré :", scytale_dechiffrer(chiffre, 7))
print()
from collections import Counter
comptes_clair = Counter(c for c in clair)
comptes_chiffre = Counter(c for c in chiffre)
print("Lettres du clair et du chiffré, triées par fréquence décroissante :")
print(" ", [c for c, _ in comptes_clair.most_common()])
print(" ", [c for c, _ in comptes_chiffre.most_common()])
print()
print("Les deux listes sont identiques : la permutation n'a rien changé aux fréquences.")

### Pourquoi est-ce une catastrophe pour le chiffreur ?

Parce que le texte chiffré a exactement les fréquences du français : un
cryptanalyste voit immédiatement qu'il a affaire à une permutation, et il lui
suffit d'essayer les clefs possibles (il y en a très peu !). Encore faut-il avoir
un moyen de reconnaître le bon candidat : c'est le rôle de la fonction
`vraisemblance`, qui note les paires de lettres consécutives (« ES », « LE », « DE »
sont fréquentes en français ; « XZ » ou « QK » ne le sont pas).

La cellule suivante montre la vraisemblance d'un texte français et celle d'un
même texte relu dans le désordre.


In [ ]:
clair = "LEGRANDMESSAGERARRIVECETTEAPRESMIDI"
melange = scytale_chiffrer(clair, 7)
print("vraisemblance du texte clair    :", round(vraisemblance(clair), 3))
print("vraisemblance du texte mélangé  :", round(vraisemblance(melange), 3))
print("(plus le nombre est grand — donc moins négatif — plus le texte ressemble à du français)")

## 6. Exercice 2 — casser la scytale (fonction B.3)

Écrivez `cle_scytale_possibles`, `decouper_en_morceaux`, puis `attaque_scytale` dans
votre fichier. Puis déchiffrez le message intercepté `MESSAGE_SCYTALE`.


In [ ]:
tp = charger()

def _exercice():
    cle, texte = tp.attaque_scytale(tp.MESSAGE_SCYTALE)
    print("Clef trouvée :", cle)
    print("Message :", texte)

executer("B.3 attaque_scytale (et B.1, B.2)", _exercice)
verifier()

## 7. Le code de César (code de substitution)

$$c = (\text{rang}(m) + k) \bmod 26$$

Cette fois, les lettres changent, mais **chaque lettre garde son rôle** : la lettre
la plus fréquente du texte chiffré est la lettre la plus fréquente du texte clair,
décalée. L'histogramme du message chiffré est donc celui du français, translaté
(comparez celui que vous avez affiché plus haut avec le corpus de référence).

### Expérience : les deux méthodes d'attaque

1. la méthode « historique » : la lettre la plus fréquente est presque toujours le E,
   donc le décalage probable est l'écart entre cette lettre et le E ;
2. la méthode automatique : la plus petite `distance_au_francais` parmi les 26 candidats.

La deuxième est celle que vous implémenterez (fonction C.3), car elle est facile à
automatiser et marche même quand le message est court.


In [ ]:
tp = charger()
from outils.reference import cesar_chiffrer

message = "LE GRAND MESSAGE ARRIVE CE SOIR, LA MISSION EST EN COURS"
for cle in (3, 12, 25):
    chiffre = cesar_chiffrer(message, cle)
    print(f"clef {cle:2d} -> {chiffre[:40]}...")
print()
print("Distances au français des 26 déchiffrements possibles du vrai message intercepté :")
for cle in range(26):
    candidat = cesar_dechiffrer(tp.MESSAGE_CESAR, cle)
    print(f"  clef {cle:2d} : distance = {distance_au_francais(candidat):9.1f}   {candidat[:38]}")

## 8. Exercice 3 — casser César (fonctions C.1 à C.3)

Écrivez `cesar_chiffrer`, `cesar_dechiffrer`, puis `forcer_cesar`, rechargez votre
fichier et déchiffrez `MESSAGE_CESAR`.


In [ ]:
tp = charger()

def _exercice():
    cle, texte = tp.forcer_cesar(tp.MESSAGE_CESAR)
    print("Clef trouvée :", cle)
    print("Message :", texte)

executer("C.3 forcer_cesar (et C.1, C.2)", _exercice)
verifier()

## 9. Exercice 4 — Marie Stuart, et l'ancêtre des codes d'authentification (D.1)

Marie Stuart (1587) correspondait avec ses partisans par un chiffre de substitution.
Les services de la reine Élisabeth, qui connaissaient le système, firent glisser dans
les messages une phrase que Marie ne pouvait que reprendre : une **lettre de la
fourche**, qui servit de preuve lors de son procès.

Pour qu'un piège fonctionne, il faut pouvoir vérifier automatiquement qu'un texte
contient bien la phrase attendue, indépendamment des majuscules, des espaces et de la
ponctuation. C'est exactement ce que fait la fonction `message_compromis` (D.1), et
c'est l'ancêtre des codes d'authentification (MAC) vus en fin de cours.


In [ ]:
tp = charger()

def _exercice():
    document = "XWWD TXH, O'DSSURFKH. ATTAQUE A L'AUBE, PREVENEZ LE BATELIER."
    print("ATTAQUE A L AUBE est-il dans le document ?", tp.message_compromis("ATTAQUE A L AUBE", document))
    print("FUITE est-il dans le document ?", tp.message_compromis("FUITE", document))

executer("D.1 message_compromis", _exercice)
verifier()

## 10. Questions de rapport

Répondez en quelques phrases, en vous appuyant sur ce que vous avez observé.

1. **Pourquoi** les fréquences des lettres d'un texte chiffré par la scytale sont-elles
   exactement celles du texte clair ? Quelle conséquence pour la cryptanalyse ?
2. La scytale avec la clef $t$ : combien y a-t-il de clefs possibles pour un message
   de $n$ lettres ? Expliquez pourquoi l'attaque par essais successifs est peu coûteuse.
3. Le code de César : combien de clefs différentes ? Combien en reste-t-il si l'on
   sait que la lettre la plus fréquente du texte chiffré est probablement un E ?
4. Comparer deux critères de « ressemblance au français » : compter les occurrences des
   petits mots (DE, LE, ES...) et calculer la vraisemblance des paires de lettres
   (fonction `vraisemblance`). Lequel est le plus fiable sur un message court ?
   Justifiez par une expérience (chiffrez une phrase de 20 lettres, attaquez-la avec
   les deux critères).
5. Qu'est-ce que le principe de Kerckhoffs, et en quoi les deux codes de cette séance
   le respectent-ils (ou non) ?
6. Un code de substitution **homophonique** (le grand chiffre de Louis XIV) remplace une
   lettre par plusieurs symboles possibles, tirés au hasard. Pourquoi cela gêne-t-il
   l'analyse de fréquences ? Quelle nouvelle difficulté de cryptanalyse fait-il apparaître ?


## 11. Pour aller plus loin (facultatif)

* **Quand l'attaque de la scytale échoue** : essayez avec un message très court
  (par exemple 12 lettres). Pourquoi la méthode devient-elle peu fiable ?
* **Une permutation beaucoup plus solide** : au lieu de lire le tableau en colonnes,
  on mélange les colonnes selon un mot de clef (transposition par colonnes, utilisée
  jusqu'au XXe siècle, et reprise dans le chapitre « Codes modernes »).
* **Le X de complément** : que se passe-t-il si le message contient naturellement de
  nombreux X ? Comment le destinataire peut-il distinguer le vrai X du X ajouté ?


In [ ]:
# Une transposition par mot de clef, pour comparaison avec la scytale.
def transposition_par_mot(texte, mot_clef):
    """Écrit le texte en lignes, puis relit les colonnes dans l'ordre alphabétique du mot de clef."""
    ordre = sorted(range(len(mot_clef)), key=lambda i: (mot_clef[i], i))
    complement = (-len(texte)) % len(mot_clef)
    tableau = texte + "X" * complement
    largeur = len(mot_clef)
    colonnes = ["".join(tableau[i] for i in range(debut, len(tableau), largeur)) for debut in range(largeur)]
    return "".join(colonnes[i] for i in ordre)

def de_transposition_par_mot(chiffre, mot_clef):
    ordre = sorted(range(len(mot_clef)), key=lambda i: (mot_clef[i], i))
    largeur = len(mot_clef)
    lignes = len(chiffre) // largeur
    colonnes = [""] * largeur
    for rang, indice in enumerate(ordre):
        colonnes[indice] = chiffre[rang * lignes:(rang + 1) * lignes]
    return "".join(colonnes[i % largeur][i // largeur] for i in range(len(chiffre)))

clair = "LEGRANDMESSAGERARRIVECETTEAPRESMIDI"
chiffre = transposition_par_mot(clair, "SECRET")
print("Clair   :", clair)
print("Chiffré :", chiffre)
print("Déchiffré :", de_transposition_par_mot(chiffre, "SECRET"))
print()
print("vraisemblance du clair  :", round(vraisemblance(clair), 3))
print("vraisemblance du chiffré:", round(vraisemblance(chiffre), 3))
print()
print("Nombre de clefs possibles pour un mot de 6 lettres :", 6 * 5 * 4 * 3 * 2 * 1)